### First few cells is just data preparation

My implementation is build on top of nequip and allegro python libraries to simplify data load and preprocessing.
The wigner simbols are from e3nn
Here are the links
https://github.com/mir-group/nequip
https://github.com/mir-group/allegro

In [1]:
# reference example
from nequip.data import dataset_from_config
from nequip.utils import Config
#from nequip.utils.misc import get_default_device_name
#from nequip.utils.config import _GLOBAL_ALL_ASKED_FOR_KEYS

from nequip.model import model_from_config
import os

default_config = dict(
    root="./",
    tensorboard=False,
    wandb=False,
    model_builders=[
        "SimpleIrrepsConfig",
        "EnergyModel",
        "PerSpeciesRescale",
        "StressForceOutput",
        "RescaleEnergyEtc",
    ],
    dataset_statistics_stride=1,
    device='cpu',
    default_dtype="float64",
    model_dtype="float32",
    allow_tf32=True,
    verbose="INFO",
    model_debug_mode=False,
    equivariance_test=False,
    grad_anomaly_mode=False,
    gpu_oom_offload=False,
    append=False,
    warn_unused=False,
    _jit_bailout_depth=2,  # avoid 20 iters of pain, see https://github.com/pytorch/pytorch/issues/52286
    # Quote from eelison in PyTorch slack:
    # https://pytorch.slack.com/archives/CDZD1FANA/p1644259272007529?thread_ts=1644064449.039479&cid=CDZD1FANA
    # > Right now the default behavior is to specialize twice on static shapes and then on dynamic shapes.
    # > To reduce warmup time you can do something like setFusionStrartegy({{FusionBehavior::DYNAMIC, 3}})
    # > ... Although we would wouldn't really expect to recompile a dynamic shape fusion in a model,
    # > provided broadcasting patterns remain fixed
    # We default to DYNAMIC alone because the number of edges is always dynamic,
    # even if the number of atoms is fixed:
    _jit_fusion_strategy=[("DYNAMIC", 3)],
    # Due to what appear to be ongoing bugs with nvFuser, we default to NNC (fuser1) for now:
    # TODO: still default to NNC on CPU regardless even if change this for GPU
    # TODO: default for ROCm?
    _jit_fuser="fuser1",
)
import numpy as np
import random
import torch
def set_seed(seed: int = 42) -> None:
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    # When running on the CuDNN backend, two further options must be set
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    # Set a fixed value for the hash seed
    os.environ["PYTHONHASHSEED"] = str(seed)
    #print(f"Random seed set as {seed}")

os.environ['NEQUIP_NUM_TASKS'] = '4'
# All default_config keys are valid / requested
#_GLOBAL_ALL_ASKED_FOR_KEYS.update(default_config.keys())

In [3]:
config = Config.from_file('./config/example_ETN_opt_MEA.yaml', defaults=default_config)

config['root'] = 'results/MEA_Allegro_0'
config['seed'] = 123456 + 8
set_seed(config['seed'])
torch.manual_seed(config['seed'])
dataset = dataset_from_config(config, prefix="dataset")

validation_dataset = None

In [4]:
# Trainer
from nequip.train.trainer import Trainer
from e3nn import o3

trainer = Trainer(model=None, **Config.as_dict(config))

# what is this
# to update wandb data?
config.update(trainer.params)

# = Train/test split =
trainer.set_dataset(dataset, validation_dataset)

# Some hyperparameteres
#Nc = 10 # number of chennels for F features from ETN paper
#N_rank_spec = 4 # hidden rank of reduction for type radial tensor
#config['Nc'] = Nc
#config['N_rank_spec'] = N_rank_spec

# ETN parameters
#config['d'] = 4 # dimention of the tensor train
#config['N_rank_ett'] = [4, 4, 4] # ranks of tensor train



# = Build model =
final_model = model_from_config(
    config=config, initialize=True, dataset=trainer.dataset_train
)

DEBUG:root:* Initialize Output
  ...generate file name results/MEA_Allegro_0/example/log
  ...open log file results/MEA_Allegro_0/example/log
  ...generate file name results/MEA_Allegro_0/example/metrics_epoch.csv
  ...open log file results/MEA_Allegro_0/example/metrics_epoch.csv
  ...generate file name results/MEA_Allegro_0/example/metrics_initialization.csv
  ...open log file results/MEA_Allegro_0/example/metrics_initialization.csv
  ...generate file name results/MEA_Allegro_0/example/metrics_batch_train.csv
  ...open log file results/MEA_Allegro_0/example/metrics_batch_train.csv
  ...generate file name results/MEA_Allegro_0/example/metrics_batch_val.csv
  ...open log file results/MEA_Allegro_0/example/metrics_batch_val.csv
  ...generate file name results/MEA_Allegro_0/example/best_model.pth
  ...generate file name results/MEA_Allegro_0/example/last_model.pth
  ...generate file name results/MEA_Allegro_0/example/trainer.pth
  ...generate file name results/MEA_Allegro_0/example/config

...PerSpeciesScaleShift_param = dict(
...   optional_args = {'out_field': 'atomic_energy', 'scales_trainable': False, 'shifts_trainable': False, 'default_dtype': 'float32', 'num_types': 4, 'type_names': ['Nb', 'Mo', 'Ta', 'W'], 'field': 'atomic_energy', 'shifts': tensor(-11.4157), 'scales': tensor(0.8584), 'arguments_in_dataset_units': True},
...   positional_args = {'irreps_in': {'pos': 1x1oe, 'edge_index': None, 'edge_types': 1x0ee, 'node_attrs': 4x0ee, 'node_features': 4x0ee, 'edge_embedding': 8x0ee, 'edge_cutoff': 1x0ee, 'edge_attrs': 1x0ee+1x1oe+1x2ee, 'edge_features_F': 10x0ee+10x1oe+10x2ee, 'node_features_F': 10x0ee+10x1oe+10x2ee, 'node_features_ETN': 10x0ee+10x1oe+10x2ee, 'atomic_energy': 1x0ee}})
Replace string dataset_forces_rms to 0.8583642840385437
Initially outputs are globally scaled by: 0.8583642840385437, total_energy are globally shifted by None.
PerSpeciesScaleShift's arguments were in dataset units; rescaling:
  Original scales: [Nb: 0.858364, Mo: 0.858364, Ta: 0.858

In [6]:
trainer.use_ema

True

In [4]:
import torch
from torch.nn.functional import one_hot
from nequip.data import AtomicData, AtomicDataDict
from torch.nn.functional import one_hot
from e3nn.nn import FullyConnectedNet
from allegro import with_edge_spin_length
from allegro import _keys
from torch import nn
import math

trainer.model = final_model

# Test configuration stores as dict of parameters
data0 = AtomicData.to_AtomicDataDict(dataset[0])

In [5]:
import torch
from torch.nn.functional import one_hot
from nequip.data import AtomicData, AtomicDataDict
from torch.nn.functional import one_hot
from e3nn.nn import FullyConnectedNet
    
from torch import nn
import math
# forward pass
data_new = final_model(data0)



In [13]:
from time import perf_counter
from nequip.utils import finish_all_writes, atomic_write_group, finish_all_writes

def init_trainer(trainer):
    """Init the trainer"""
    
    if not trainer._initialized:
        trainer.init()

    for callback in trainer._init_callbacks:
        callback(trainer)

    trainer.init_log()
    trainer.wall = perf_counter()
    trainer.previous_cumulative_wall = trainer.cumulative_wall
    
    with atomic_write_group():
        if trainer.iepoch == -1:
            trainer.save()
        if trainer.iepoch in [-1, 0]:
            trainer.save_config()
    
    trainer.init_metrics()

def run_one_sweep_cycle(trainer, config):
    
    cur_sweep = 0 
    # Making all non trainable
    for i in range(config['d']):
        trainer.model.get_submodule('model.model.func.etn.cores')[i].requires_grad_(False)
    
    # Forward sweeps
    for i in range(config['d']):
        trainer.model.get_submodule('model.model.func.etn.cores')[i].requires_grad_(True)
        
        if i != 0:
            trainer.model.get_submodule('model.model.func.etn.cores')[i-1].requires_grad_(False)

        for j in range(config['epochs_per_sweep']):
            trainer.epoch_step()
            trainer.end_of_epoch_save()
        
        cur_sweep += 1
    
    # Backward sweeps   
    for i in range(config['d']-2, -1, -1):
        trainer.model.get_submodule('model.model.func.etn.cores')[i].requires_grad_(True)
        trainer.model.get_submodule('model.model.func.etn.cores')[i+1].requires_grad_(False)
        
        for j in range(config['epochs_per_sweep']):
            trainer.epoch_step()
            trainer.end_of_epoch_save()

        cur_sweep += 1
        
    return cur_sweep 


In [41]:
config['max_epochs'] = 2
config['epochs_per_sweep'] = 1

In [46]:
ind = 2

config['root'] = f'results/MEA_Allegro_{ind}'
config['seed'] = 1234560 + ind

dataset = dataset_from_config(config, prefix="dataset")
validation_dataset = None    

# Trainer
trainer = Trainer(model=None, **Config.as_dict(config))

# what is this
# to update wandb data?
config.update(trainer.params)

# = Train/test split =
trainer.set_dataset(dataset, validation_dataset)

# = Build model =
final_model = model_from_config(
    config=config, initialize=True, dataset=trainer.dataset_train)


trainer.model = final_model

#trainer.train()

# Init the trainer
init_trainer(trainer)

# Check if epoch per sweep has a correct value
assert (config['max_epochs'] % config['epochs_per_sweep'] == 0)


num_sweeps = config['max_epochs'] // config['epochs_per_sweep']

cur_sweep = 0
while not trainer.stop_cond and cur_sweep < num_sweeps:
    cur_sweep += 1
    trainer.epoch_step()
    trainer.end_of_epoch_save()
    trainer.epoch_step()
    trainer.end_of_epoch_save()
    #cur_sweep += run_one_sweep_cycle(trainer, config)

#trainer.iepoch = config['max_epochs']
trainer.stop_cond

for callback in trainer._final_callbacks:
    callback(trainer)

trainer.final_log()

trainer.save()
finish_all_writes()

search for AtomicData_options with prefix dataset
          0_args :                                  AtomicData_options <-                         dataset_AtomicData_options
search for r_max with prefix dataset
          1_args :                                               r_max
instantiate TypeMapper
   optional_args :                             type_to_chemical_symbol
   optional_args :                             chemical_symbol_to_type
   optional_args :                                          type_names
...TypeMapper_param = dict(
...   optional_args = {'type_names': ['Nb', 'Mo', 'Ta', 'W'], 'chemical_symbol_to_type': {'Nb': 0, 'Mo': 1, 'Ta': 2, 'W': 3}, 'type_to_chemical_symbol': {0: 'Nb', 1: 'Mo', 2: 'Ta', 3: 'W'}, 'chemical_symbols': None},
...   positional_args = {})
instantiate register_fields
...register_fields_param = dict(
...   optional_args = {'node_fields': [], 'edge_fields': [], 'graph_fields': [], 'long_fields': []},
...   positional_args = {})
instantiate ASEDat

In [47]:
trainer.iepoch

3

In [20]:
trainer.stop_cond += 

False

In [10]:
trainer.model.get_submodule('model.model.func.etn.cores')

ParameterList(
    (0): Parameter containing: [torch.float32 of size 3x1x10x4]
    (1): Parameter containing: [torch.float32 of size 11x4x10x4]
    (2): Parameter containing: [torch.float32 of size 11x4x10x4]
    (3): Parameter containing: [torch.float32 of size 3x4x10x1]
)

In [ ]:
  Train      #    Epoch      wal       LR       loss_f       loss_e         loss        f_mae       f_rmse        e_mae      e/N_mae
! Train               2   52.071    0.001       0.0737       0.0235       0.0972         0.14        0.248         2.13       0.0815
! Validation          2   52.071    0.001       0.0601       0.0118       0.0719        0.137        0.238          1.4       0.0518
Wall time: 52.073412504047155
! Best model        2    0.072

In [22]:
trainer.model.get_submodule('model.model.func.etn.cores')[-1][0, 0, 2, 0]

tensor(0.9773)

In [31]:
config['max_epochs']

2

In [12]:
trainer.model.get_submodule('model.model.func.etn.cores')[0]

Parameter containing:
tensor([[[[-7.1772e-01, -7.8159e-01, -1.5125e+00,  2.4494e-01],
          [-1.2428e+00, -1.1642e+00,  6.2678e-01, -1.1915e-02],
          [ 3.4441e-02, -1.4269e+00, -1.4098e+00,  4.3609e-01],
          [ 1.4143e+00, -2.1563e+00, -7.8891e-01,  7.9075e-01],
          [ 8.7655e-01,  9.7975e-02,  1.0961e+00, -2.0706e-01],
          [ 5.0269e-01,  1.3863e+00, -2.5744e-01, -6.5676e-01],
          [-1.8227e+00, -7.1112e-01, -1.8864e+00, -9.3011e-01],
          [ 3.2355e-01, -6.8184e-01, -6.7032e-02,  1.3290e+00],
          [-1.6632e+00, -2.0090e+00, -2.0004e-01,  7.4554e-01],
          [-1.1288e+00, -7.8105e-01, -1.7076e+00, -1.3615e+00]]],


        [[[-2.1701e-01, -1.5466e+00,  1.0975e+00, -1.8001e+00],
          [-1.1987e+00,  1.2430e+00, -5.0787e-01,  9.5395e-01],
          [-1.1154e-01,  1.9087e+00, -9.5710e-01,  2.9477e-01],
          [-3.8304e-01,  1.0747e+00,  2.7312e-01, -6.1511e-01],
          [ 3.7556e-01,  1.9222e-01, -1.8732e+00, -2.2219e-02],
          [-2.

In [16]:
from nequip.utils import atomic_write_group
from nequip.utils import finish_all_writes
from time import perf_counter

# Number of sweeps
nsw = 20
epochs_per_sweep = 1
assert (config['max_epochs'] % config['epochs_per_sweep'] == 0)
cur_sweep = 0


if not trainer._initialized:
    trainer.init()

for callback in trainer._init_callbacks:
    callback(self)

trainer.init_log()
trainer.wall = perf_counter()
trainer.previous_cumulative_wall = trainer.cumulative_wall

with atomic_write_group():
    if trainer.iepoch == -1:
        trainer.save()
    if trainer.iepoch in [-1, 0]:
        trainer.save_config()

trainer.init_metrics()

if getattr(trainer, "_post_init_callback", None) is not None:
    trainer._post_init_callback()


while cur_sweep < nsw:
    # Making all non trainable
    for i in range(config['d']):
        trainer.model.get_submodule('model.model.func.etn.cores')[i].requires_grad_(False)
    
    # Forward sweeps
    for i in range(config['d']):
        trainer.model.get_submodule('model.model.func.etn.cores')[i].requires_grad_(True)
        
        if i != 0:
            trainer.model.get_submodule('model.model.func.etn.cores')[i-1].requires_grad_(False)
        
        trainer.epoch_step()
        trainer.end_of_epoch_save()
        
        cur_sweep += 1
    
    # Backward sweeps   
    for i in range(config['d']-2, -1, -1):
        trainer.model.get_submodule('model.model.func.etn.cores')[i].requires_grad_(True)
        trainer.model.get_submodule('model.model.func.etn.cores')[i+1].requires_grad_(False)
        
        cur_sweep += 1
        trainer.epoch_step()
        trainer.end_of_epoch_save()

trainer.epoch_step()
trainer.end_of_epoch_save()

! Starting training ...
Saved trainer to results/MEA_Allegro_2/example/trainer.pth
Saved last model to to results/MEA_Allegro_2/example/last_model.pth
instantiate Metrics
...Metrics_param = dict(
...   optional_args = {},
...   positional_args = {'components': [['forces', 'mae'], ['forces', 'rmse'], ['total_energy', 'mae'], ['total_energy', 'mae', {'PerAtom': True}]]})
instantiate L1Loss
...L1Loss_param = dict(
...   optional_args = {'size_average': None, 'reduce': None},
...   positional_args = {'reduction': 'none'})
instantiate L1Loss
...L1Loss_param = dict(
...   optional_args = {'size_average': None, 'reduce': None},
...   positional_args = {'reduction': 'none'})
instantiate L1Loss
...L1Loss_param = dict(
...   optional_args = {'size_average': None, 'reduce': None},
...   positional_args = {'reduction': 'none'})
instantiate L1Loss
...L1Loss_param = dict(
...   optional_args = {'size_average': None, 'reduce': None},
...   positional_args = {'reduction': 'none'})

validation
# Epoch 

In [14]:
from nequip.utils import finish_all_writes
from time import perf_counter

for callback in trainer._final_callbacks:
    callback(trainer)

trainer.final_log()

trainer.save()
finish_all_writes()

AttributeError: 'Trainer' object has no attribute 'stop_arg'

In [11]:

trainer.epoch_step()
trainer.end_of_epoch_save()


training
# Epoch batch         loss       loss_f       loss_e        f_mae       f_rmse        e_mae      e/N_mae
      5   100        0.153        0.145      0.00786        0.191        0.327          1.2       0.0547
      5   200       0.0489       0.0237       0.0252       0.0753        0.132         2.74       0.0597
      5   300      0.00897       0.0057      0.00327       0.0406       0.0648        0.458       0.0333
      5   400       0.0131       0.0101      0.00299       0.0559       0.0862        0.647       0.0419
      5   500       0.0356       0.0297      0.00592       0.0834        0.148         1.37        0.043

validation
# Epoch batch         loss       loss_f       loss_e        f_mae       f_rmse        e_mae      e/N_mae
      5   100       0.0423       0.0406      0.00165        0.111        0.173        0.544       0.0259


  Train      #    Epoch      wal       LR       loss_f       loss_e         loss        f_mae       f_rmse        e_mae      e/N_mae
! T

In [10]:
def batch_step(data, validation=False):
    # no need to have gradients from old steps taking up memory
    #self.optim.zero_grad(set_to_none=True)

    #if validation:
    #    self.model.eval()
    #else:
    #    self.model.train()

    # Do any target rescaling
    data = AtomicData.to_AtomicDataDict(data)

    # this will normalize the targets
    # in both validation and train we want targets normalized _for the loss_
    data_for_loss = trainer.model.unscale(data, force_process=True)

    # Run model
    # We make a shallow copy of the input dict in case the model modifies it
    out = trainer.model(data_for_loss)
    #print(out)
    return out

In [12]:
from nequip.train._key import ABBREV, LOSS_KEY, TRAIN, VALIDATION

def epoch_step(trainer):

    dataloaders = {TRAIN: trainer.dl_train, VALIDATION: trainer.dl_val}
    categories = [TRAIN, VALIDATION] if trainer.iepoch >= 0 else [VALIDATION]
    dataloaders = [
        dataloaders[c] for c in categories
    ]  # get the right dataloaders for the catagories we actually run
    if TRAIN in categories:
        # We have to step the sampler so it knows what epoch it is
        trainer.dl_train_sampler.step_epoch(trainer.iepoch)

    #self.metrics_dict = {}
    #self.loss_dict = {}

    for category, dataset in zip(categories, dataloaders):
        
        for trainer.ibatch, batch in enumerate(dataset):
            print(trainer.ibatch, batch)
            out = batch_step(
                data=batch,
                validation=(category == VALIDATION),
            )
            
    return out, batch

In [13]:
len(dataset[:20])

20

In [14]:
dataset[:20]

ASEDataset(20)

In [15]:
trainer.n_train

5000

In [16]:
trainer.dl_val.batch_size

5

In [19]:
trainer.n_train = 2
trainer.n_val = 3

trainer.train_idcs = torch.tensor([201, 1201], dtype = torch.long)
trainer.val_idcs = torch.tensor([301, 1401, 5], dtype = torch.long)

trainer.set_dataset(dataset, None)

In [20]:
out, batch = epoch_step(trainer)

0 Batch(atom_types=[34, 1], batch=[34], cell=[3, 3, 3], edge_cell_shift=[884, 3], edge_index=[2, 884], forces=[34, 3], pbc=[3, 3], pos=[34, 3], ptr=[4], stress=[3, 3, 3], total_energy=[3, 1])


In [21]:
out['pos'].shape

torch.Size([34, 3])

In [22]:
out['batch']

tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 2, 2])

In [23]:
out['ptr']

tensor([ 0, 16, 32, 34])

In [24]:
print(AtomicData.to_AtomicDataDict(dataset[301])['pos'].shape)
print(AtomicData.to_AtomicDataDict(dataset[1401])['pos'].shape)
print(AtomicData.to_AtomicDataDict(dataset[5])['pos'].shape)

torch.Size([16, 3])
torch.Size([16, 3])
torch.Size([2, 3])


In [25]:
out.keys()

dict_keys(['edge_index', 'pos', 'batch', 'ptr', 'cell', 'edge_cell_shift', 'atom_types', 'edge_vectors', 'edge_types', 'node_attrs', 'node_features', 'edge_lengths', 'edge_embedding', 'edge_cutoff', 'edge_attrs', 'edge_features_F', 'node_features_F', 'node_features_ETN', 'atomic_energy', 'total_energy', 'forces', 'stress', 'virial', 'atom_virial'])

In [28]:
trainer.batch_metrics = trainer.metrics(pred=out, ref=batch)

ValueError: Data shape of batch, torch.Size([3, 34]), does not match the input data dimension of this RunningStats, torch.Size([192])

In [29]:
batch

Batch(atom_types=[34, 1], batch=[34], cell=[3, 3, 3], edge_cell_shift=[884, 3], edge_index=[2, 884], forces=[34, 3], pbc=[3, 3], pos=[34, 3], ptr=[4], stress=[3, 3, 3], total_energy=[3, 1])

In [30]:
for key in out:
    print(key, out[key].shape)

edge_index torch.Size([2, 884])
pos torch.Size([34, 3])
batch torch.Size([34])
ptr torch.Size([4])
cell torch.Size([3, 3, 3])
edge_cell_shift torch.Size([884, 3])
atom_types torch.Size([34, 1])
edge_vectors torch.Size([884, 3])
edge_types torch.Size([884, 1])
node_attrs torch.Size([34, 4])
node_features torch.Size([34, 4])
edge_lengths torch.Size([884])
edge_embedding torch.Size([884, 8])
edge_cutoff torch.Size([884, 1])
edge_attrs torch.Size([884, 9])
edge_features_F torch.Size([884, 9, 10])
node_features_F torch.Size([34, 9, 10])
node_features_ETN torch.Size([34, 9, 10])
atomic_energy torch.Size([34, 34])
total_energy torch.Size([3, 34])
forces torch.Size([34, 3])
stress torch.Size([3, 3, 3])
virial torch.Size([3, 3, 3])
atom_virial torch.Size([34, 3, 3])


In [31]:
batch

Batch(atom_types=[34, 1], batch=[34], cell=[3, 3, 3], edge_cell_shift=[884, 3], edge_index=[2, 884], forces=[34, 3], pbc=[3, 3], pos=[34, 3], ptr=[4], stress=[3, 3, 3], total_energy=[3, 1])

In [32]:
out['total_energy'].shape

torch.Size([3, 34])

In [40]:

( data_new[_keys.NODE_FEATURES_ETN] * data_new[_keys.NODE_FEATURES_ETN] ).sum(dim = (-2, -1)).unsqueeze(-1)

tensor([[5.7274e-15],
        [5.7274e-15]], grad_fn=<UnsqueezeBackward0>)

In [37]:
data_new["node_features_F"].shape

torch.Size([2, 9, 10])